# Klasifikasi DemogPairs Menggunakan ViT (Emosi dan Umur) & Random Forest

In [1]:
import numpy as np
import utils as u
import joblib
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, ParameterGrid
from imblearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from tqdm import tqdm

joblib.parallel_backend('threading')

## Load Dataset

In [2]:
data = u.load_demogpairs()
pd.DataFrame(data)

,db_code,image_path,full_path,label,label_idx
0,CWF,able_wanamakok/002.jpg,dataset/demogpairs/images\able_wanamakok/002.jpg,Asian_Females,5
1,CWF,able_wanamakok/004.jpg,dataset/demogpairs/images\able_wanamakok/004.jpg,Asian_Females,5
2,CWF,able_wanamakok/007.jpg,dataset/demogpairs/images\able_wanamakok/007.jpg,Asian_Females,5
3,CWF,able_wanamakok/008.jpg,dataset/demogpairs/images\able_wanamakok/008.jpg,Asian_Females,5
4,CWF,able_wanamakok/012.jpg,dataset/demogpairs/images\able_wanamakok/012.jpg,Asian_Females,5
...,...,...,...,...,...
10795,CWF,zachary_quinto/177.jpg,dataset/demogpairs/images\zachary_quinto/177.jpg,White_Males,3
10796,CWF,zachary_quinto/214.jpg,dataset/demogpairs/images\zachary_quinto/214.jpg,White_Males,3
10797,CWF,zachary_quinto/217.jpg,dataset/demogpairs/images\zachary_quinto/217.jpg,White_Males,3
10798,CWF,zachary_quinto/218.jpg,dataset/demogpairs/images\zachary_quinto/218.jpg,White_Males,3


## Load Fitur

In [3]:
emotion_features = joblib.load('features/demogpairs_vit-emotion.pkl')
age_features = joblib.load('features/demogpairs_vit-age.pkl')
features = {}
for d in tqdm(data):
    key = d['image_path']
    features[key] = np.array(list(emotion_features[key]) + list(age_features[key]))
print('Jumlah fitur per gambar:', np.array(features[list(features.keys())[0]]).shape[0])

  0%|          | 0/10800 [00:00<?, ?it/s]

  6%|▌         | 616/10800 [00:00<00:01, 6151.54it/s]

 11%|█▏        | 1232/10800 [00:00<00:01, 5389.16it/s]

 16%|█▋        | 1782/10800 [00:00<00:01, 5433.16it/s]

 22%|██▏       | 2329/10800 [00:00<00:01, 5197.09it/s]

...

 90%|█████████ | 9732/10800 [00:01<00:00, 5169.55it/s]

 95%|█████████▍| 10252/10800 [00:01<00:00, 4808.64it/s]

100%|█████████▉| 10776/10800 [00:02<00:00, 4926.73it/s]

100%|██████████| 10800/10800 [00:02<00:00, 5283.34it/s]

Jumlah fitur per gambar: 1536


## Split Data

In [4]:
X = np.array([features[d['image_path']] for d in data])
y = np.array([d['label_idx'] for d in data])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print((len(X_train), len(X_test)))

(8640, 2160)


## Kombinasi Parameter

In [5]:
grid_params = [
    {
        'scaler': [None, MinMaxScaler()],
        'pca': [None, PCA(n_components=0.5), PCA(n_components=0.75)],
        
        'classifier': [RandomForestClassifier(random_state=42)],
        'classifier__n_estimators': [100, 200],
        'classifier__max_depth': [None, 20, 30],
        'classifier__min_samples_split': [2, 5],
        'classifier__min_samples_leaf': [1, 2],
        'classifier__max_features': ['sqrt', 'log2'],
    },
]

pipeline = Pipeline(steps=[
    ('scaler', None),
    ('pca', None),
    ('classifier', None)
])

skv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    'accuracy': 'accuracy', 
    'f1': 'f1_macro', 
    'precision': 'precision_macro', 
    'recall': 'recall_macro',
}

grid_models = {}
for params in grid_params:
    key = str(params['classifier'][0]).split('(')[0]
    grid_models[key] = GridSearchCV(
        estimator=pipeline,
        param_grid=params,
        cv=skv, refit='accuracy',
        scoring=scoring, n_jobs=int(joblib.cpu_count() * 0.6),
        verbose=1, error_score='raise',
        return_train_score=True
    )
    print(f'{key}: {len(ParameterGrid(params))} kombinasi')

RandomForestClassifier: 288 kombinasi


## Klasifikasi

In [6]:
evaluation_results, fold_results = u.evaluate_models(
    grid_models, 
    X_train, y_train,
    X_test, y_test,
    target_names=u.demogpairs_classes,
    model_prefix="models/clf_demogpairs_rf_vit-emotion-age_",
    results_path="results/demogpairs_rf_vit-emotion-age_"
)
sorted_results = pd.DataFrame(evaluation_results).sort_values(by="test_accuracy", ascending=False).to_dict("records")
u.html_br()
_dtable = u.display_table(sorted_results)

Evaluating: RandomForestClassifier


{'classifier': 'RandomForestClassifier', 'classifier__max_depth': None, 'classifier__max_features': 'log2', 'classifier__min_samples_leaf': 2, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 200, 'pca': 'PCA', 'scaler': None}


Accuracy  : 0.8111111111111111
Precision : 0.8111192873921357
Recall    : 0.8111111111111112
F1 Score  : 0.8107774784200021
               precision    recall  f1-score   support

Asian_Females     0.7732    0.7861    0.7796       360
  Asian_Males     0.7884    0.7556    0.7716       360
Black_Females     0.8195    0.7944    0.8068       360
  Black_Males     0.8497    0.8639    0.8567       360
White_Females     0.8271    0.7972    0.8119       360
  White_Males     0.8088    0.8694    0.8380       360

     accuracy                         0.8111      2160
    macro avg     0.8111    0.8111    0.8108      2160
 weighted avg     0.8111    0.8111    0.8108      2160



Class,OvR Accuracy,Precision,Recall,F1-Score,Support
Asian_Females,0.9259259259259259,0.773224043715847,0.7861111111111111,0.7796143250688704,360
Asian_Males,0.9254629629629629,0.7884057971014493,0.7555555555555555,0.7716312056737589,360
Black_Females,0.9365740740740741,0.8194842406876791,0.7944444444444444,0.8067700987306065,360
Black_Males,0.9518518518518518,0.8497267759562842,0.8638888888888889,0.856749311294766,360
White_Females,0.9384259259259259,0.8270893371757925,0.7972222222222223,0.8118811881188119,360
White_Males,0.9439814814814815,0.8087855297157622,0.8694444444444445,0.8380187416331994,360


Confusion matrix saved: images\cm_rf_vit-emotion-age_RandomForestClassifier.png



Confusion Matrix:
                         Asian_Females       Asian_Males     Black_Females       Black_Males     White_Females       White_Males
       Asian_Females               283                27                16                 2                26                 6
         Asian_Males                32               272                 2                28                 2                24
       Black_Females                22                 9               286                14                22                 7
         Black_Males                 1                10                16               311                 1                21
       White_Females                27                 4                26                 0               287                16
         White_Males                 1                23                 3                11                 9               313


model_name,model_file_path,best_parameters,test_accuracy,test_f1,test_precision,test_recall,parameter_combinations
RandomForestClassifier,models/clf_demogpairs_rf_vit-emotion-age_RandomForestClassifier.pkl,"{'classifier': 'RandomForestClassifier', 'classifier__max_depth': None, 'classifier__max_features': 'log2', 'classifier__min_samples_leaf': 2, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 200, 'pca': 'PCA', 'scaler': None}",0.8111111111111111,0.8107774784200021,0.8111192873921357,0.8111111111111112,288


In [7]:
model, training_time = u.load_object('models/clf_demogpairs_rf_vit-emotion-age_RandomForestClassifier.pkl')
u.h(5, 'Waktu Pelatihan (Jobs)')
u.seconds_to_time(round(training_time))

{'input_seconds': 6282.0,
 'days': 0,
 'hours': 1,
 'minutes': 44,
 'seconds': 42.0,
 'text': '0 hari 1 jam 44 menit 42.0 detik'}

In [8]:
u.h(5, 'Waktu Pelatihan')
times = [fr['Train Time Mean'] * 5 for fr in fold_results]
u.seconds_to_time(round(np.sum(times) + model.refit_time_))

{'input_seconds': 24124.0,
 'days': 0,
 'hours': 6,
 'minutes': 42,
 'seconds': 4.0,
 'text': '0 hari 6 jam 42 menit 4.0 detik'}

In [9]:
_dtable = u.display_table(fold_results, n_items=[4, 4], column_widths=['5%', '45%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%'])

No,Params,Fold 1,Fold 2,Fold 3,Fold 4,Fold 5,Accuracy Mean,F1 Score Mean,Precision Mean,Recall Mean,Train Time Mean
1,"{'classifier': 'RandomForestClassifier', 'classifier__max_depth': None, 'classifier__max_features': 'log2', 'classifier__min_samples_leaf': 2, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 200, 'pca': 'PCA', 'scaler': None}",0.7905,0.8061,0.7928,0.8032,0.8108,0.8007,0.8006,0.8015,0.8007,17.3441
2,"{'classifier': 'RandomForestClassifier', 'classifier__max_depth': 30, 'classifier__max_features': 'log2', 'classifier__min_samples_leaf': 2, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 200, 'pca': 'PCA', 'scaler': None}",0.7905,0.8044,0.7934,0.8032,0.8108,0.8005,0.8003,0.8012,0.8005,17.9037
3,"{'classifier': 'RandomForestClassifier', 'classifier__max_depth': 30, 'classifier__max_features': 'log2', 'classifier__min_samples_leaf': 2, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 200, 'pca': 'PCA', 'scaler': None}",0.7934,0.7969,0.7905,0.8137,0.8056,0.8,0.7997,0.8004,0.8,17.6663
4,"{'classifier': 'RandomForestClassifier', 'classifier__max_depth': None, 'classifier__max_features': 'log2', 'classifier__min_samples_leaf': 2, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 200, 'pca': 'PCA', 'scaler': None}",0.7928,0.7969,0.7905,0.8137,0.8044,0.7997,0.7994,0.8001,0.7997,17.7891
...,...,...,...,...,...,...,...,...,...,...,...
285,"{'classifier': 'RandomForestClassifier', 'classifier__max_depth': 20, 'classifier__max_features': 'log2', 'classifier__min_samples_leaf': 2, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 100, 'pca': None, 'scaler': 'MinMaxScaler'}",0.7373,0.7326,0.7124,0.7199,0.7321,0.7269,0.7268,0.7295,0.7269,6.9972
286,"{'classifier': 'RandomForestClassifier', 'classifier__max_depth': 20, 'classifier__max_features': 'log2', 'classifier__min_samples_leaf': 2, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 100, 'pca': None, 'scaler': None}",0.7373,0.7326,0.7124,0.7199,0.7321,0.7269,0.7268,0.7295,0.7269,5.8863
287,"{'classifier': 'RandomForestClassifier', 'classifier__max_depth': 20, 'classifier__max_features': 'log2', 'classifier__min_samples_leaf': 2, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 100, 'pca': None, 'scaler': None}",0.7361,0.728,0.713,0.7234,0.7338,0.7269,0.7267,0.7293,0.7269,6.0834
288,"{'classifier': 'RandomForestClassifier', 'classifier__max_depth': 20, 'classifier__max_features': 'log2', 'classifier__min_samples_leaf': 2, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 100, 'pca': None, 'scaler': 'MinMaxScaler'}",0.7361,0.728,0.713,0.7234,0.7338,0.7269,0.7267,0.7293,0.7269,7.9403
